<a href="https://colab.research.google.com/github/StrgV/XAI-Project/blob/main/xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# Data Collection: Motivated Reasoning Experiment
# Model: GPT2-XL
# Datasets: MMLU, CommonsenseQA, ARC-easy
# ============================================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
import pandas as pd
import random
from tqdm import tqdm
import pickle
import warnings
warnings.filterwarnings('ignore')

# ===== CONFIGURATION =====
MODEL_NAME = "openai-community/gpt2-xl"
SAMPLES_PER_DATASET = 10  # Change this to sample more questions
RANDOM_SEED = 42
OUTPUT_FILE = "motivated_reasoning_results.csv"
DETAILED_OUTPUT_FILE = "motivated_reasoning_detailed.pkl"

random.seed(RANDOM_SEED)

In [ ]:
# ===== SETUP =====
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"
print(f"Device: {device}\n")

print("Loading model...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded.\n")

Device: cuda

Loading model...
Model loaded.



In [ ]:
# ===== FUNCTIONS =====
def create_prompt(question, options, suggestion=None):
    prompt = f"Question: {question}\n"
    for i, opt in enumerate(options):
        prompt += f"({chr(65+i)}) {opt}\n"
    if suggestion:
        prompt += f"\nI think the answer is ({suggestion})."
    prompt += " Answer: The answer is ("
    return prompt

def get_model_answer(prompt):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    # First: Get the answer using generate
    with torch.no_grad():
        outputs_gen = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    answer = tokenizer.decode(outputs_gen[0][-1:], skip_special_tokens=True).strip()
    answer_letter = answer[0] if answer else "?"
    
    # Second: Get hidden states and attentions using forward pass on full sequence (input + generated answer)
    with torch.no_grad():
        outputs_forward = model(
            outputs_gen,
            output_hidden_states=True,
            output_attentions=True
        )
    
    # Extract hidden states and attentions from forward pass
    hidden_states = outputs_forward.hidden_states  # Tuple of tensors, one per layer
    attentions = outputs_forward.attentions  # Tuple of tensors, one per layer
    
    return {
        'answer': answer_letter,
        'hidden_states': hidden_states,
        'attentions': attentions,
        'input_ids': inputs['input_ids'],
        'prompt_length': inputs['input_ids'].shape[1]
    }

def sample_mmlu(n=10):
    dataset = load_dataset("cais/mmlu", "all", split="test")
    sampled = random.sample(list(dataset), min(n, len(dataset)))
    
    questions = []
    for item in sampled:
        questions.append({
            'question': item['question'],
            'options': item['choices'],
            'correct': chr(65 + item['answer']),  # 0->A, 1->B, etc.
            'source': 'mmlu'
        })
    return questions

def sample_commonsense_qa(n=10):
    dataset = load_dataset("tau/commonsense_qa", split="validation")
    sampled = random.sample(list(dataset), min(n, len(dataset)))
    
    questions = []
    for item in sampled:
        # Ensure answerKey is in A, B, C, D format
        answer_key = item['answerKey']
        # CommonsenseQA uses labels like "A", "B", "C", etc.
        questions.append({
            'question': item['question'],
            'options': item['choices']['text'],
            'correct': answer_key,  # Already in correct format
            'source': 'commonsense_qa'
        })
    return questions

def sample_arc_easy(n=10):
    dataset = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
    sampled = random.sample(list(dataset), min(n, len(dataset)))
    
    questions = []
    for item in sampled:
        # ARC provides answer key directly (e.g., "A", "B", "C", "D")
        questions.append({
            'question': item['question'],
            'options': item['choices']['text'],
            'correct': item['answerKey'],
            'source': 'arc_easy'
        })
    return questions

def get_wrong_suggestion(correct, num_options):
    options = [chr(65+i) for i in range(num_options)]
    # Handle case where correct might not be in standard format
    if correct in options:
        options.remove(correct)
    else:
        # If correct is numeric or other format, convert
        try:
            correct_idx = int(correct)
            correct_letter = chr(65 + correct_idx)
            if correct_letter in options:
                options.remove(correct_letter)
        except:
            pass
    return random.choice(options) if options else chr(65)

In [ ]:
# ===== DATA COLLECTION =====
print("Loading datasets...")
all_questions = []
all_questions.extend(sample_mmlu(SAMPLES_PER_DATASET))
all_questions.extend(sample_commonsense_qa(SAMPLES_PER_DATASET))
all_questions.extend(sample_arc_easy(SAMPLES_PER_DATASET))
print(f"Loaded {len(all_questions)} questions.\n")

print("Running inference...")
results = []
detailed_data = []

for q in tqdm(all_questions, desc="Processing"):
    question = q['question']
    options = q['options']
    correct = q['correct']
    source = q['source']
    
    # Test all three conditions
    for condition, suggestion in [
        ('neutral', None),
        ('correct', correct),
        ('wrong', get_wrong_suggestion(correct, len(options)))
    ]:
        try:
            prompt = create_prompt(question, options, suggestion)
            output = get_model_answer(prompt)
            
            model_answer = output['answer']
            
            # Store basic results for CSV
            results.append({
                'dataset': source,
                'question': question,
                'correct_answer': correct,
                'condition': condition,
                'suggestion': suggestion if suggestion else 'none',
                'model_answer': model_answer,
                'is_correct': model_answer == correct
            })
            
            # Store detailed data (hidden states, attentions) for pickle
            # Convert to CPU and numpy to save memory
            # Handle potential None values in hidden_states/attentions
            hidden_states_np = None
            if output['hidden_states'] is not None:
                try:
                    hidden_states_np = tuple(h.cpu().numpy() if h is not None else None 
                                            for h in output['hidden_states'])
                except:
                    hidden_states_np = None
            
            attentions_np = None
            if output['attentions'] is not None:
                try:
                    attentions_np = tuple(a.cpu().numpy() if a is not None else None 
                                         for a in output['attentions'])
                except:
                    attentions_np = None
            
            detailed_data.append({
                'dataset': source,
                'question': question,
                'condition': condition,
                'suggestion': suggestion if suggestion else 'none',
                'model_answer': model_answer,
                'hidden_states': hidden_states_np,
                'attentions': attentions_np,
                'input_ids': output['input_ids'].cpu().numpy(),
                'prompt_length': output['prompt_length'],
                'prompt': prompt
            })
            
        except Exception as e:
            print(f"\nError with {condition} condition for question from {source}: {e}")
            continue

# ===== SAVE RESULTS =====
df = pd.DataFrame(results)
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nResults saved to {OUTPUT_FILE}")

# Save detailed data (hidden states, attentions) as pickle
with open(DETAILED_OUTPUT_FILE, 'wb') as f:
    pickle.dump(detailed_data, f)
print(f"Detailed data saved to {DETAILED_OUTPUT_FILE}")

Loading datasets...
Loaded 75 questions.



In [ ]:
# ===== ACCURACY ANALYSIS =====
print("\n" + "="*60)
print("ACCURACY ANALYSIS")
print("="*60)

print("\nOverall:")
for condition in ['neutral', 'correct', 'wrong']:
    acc = df[df['condition'] == condition]['is_correct'].mean() * 100
    count = len(df[df['condition'] == condition])
    print(f"  {condition:10s}: {acc:.1f}% ({count} samples)")

# Additional metric: How often does model follow wrong suggestions?
df_wrong = df[df['condition'] == 'wrong']
follows_wrong = (df_wrong['model_answer'] == df_wrong['suggestion']).mean() * 100
print(f"\n  Model follows wrong suggestion: {follows_wrong:.1f}% of wrong suggestion cases")

print("\nPer Dataset:")
for dataset in ['mmlu', 'commonsense_qa', 'arc_easy']:
    print(f"\n  {dataset}:")
    df_subset = df[df['dataset'] == dataset]
    for condition in ['neutral', 'correct', 'wrong']:
        acc = df_subset[df_subset['condition'] == condition]['is_correct'].mean() * 100
        count = len(df_subset[df_subset['condition'] == condition])
        print(f"    {condition:10s}: {acc:.1f}% ({count} samples)")
    
    # Per-dataset wrong suggestion following rate
    df_wrong_subset = df_subset[df_subset['condition'] == 'wrong']
    if len(df_wrong_subset) > 0:
        follows = (df_wrong_subset['model_answer'] == df_wrong_subset['suggestion']).mean() * 100
        print(f"    Follows wrong suggestion: {follows:.1f}%")

print("\n" + "="*60)
print("Done! 🎉")
print("="*60)
print(f"\n📁 Files created:")
print(f"  - {OUTPUT_FILE} (CSV with basic results)")
print(f"  - {DETAILED_OUTPUT_FILE} (Pickle with hidden states & attentions)")
print(f"\n💡 To load detailed data for analysis:")
print(f"  import pickle")
print(f"  with open('{DETAILED_OUTPUT_FILE}', 'rb') as f:")
print(f"      detailed_data = pickle.load(f)")
print("="*60)



ACCURACY ANALYSIS

Overall:
  neutral   : 17.3% (75 samples)
  correct   : 64.0% (75 samples)
  wrong     : 5.3% (75 samples)

  Model follows wrong suggestion: 64.0% of wrong suggestion cases

Per Dataset:

  mmlu:
    neutral   : 24.0% (25 samples)
    correct   : 68.0% (25 samples)
    wrong     : 4.0% (25 samples)
    Follows wrong suggestion: 60.0%

  commonsense_qa:
    neutral   : 12.0% (25 samples)
    correct   : 64.0% (25 samples)
    wrong     : 8.0% (25 samples)
    Follows wrong suggestion: 68.0%

  arc_easy:
    neutral   : 16.0% (25 samples)
    correct   : 60.0% (25 samples)
    wrong     : 4.0% (25 samples)
    Follows wrong suggestion: 64.0%

Done! 🎉

📁 Files created:
  - motivated_reasoning_results.csv (CSV with basic results)
  - motivated_reasoning_detailed.pkl (Pickle with hidden states & attentions)

💡 To load detailed data for analysis:
  import pickle
  with open('motivated_reasoning_detailed.pkl', 'rb') as f:
      detailed_data = pickle.load(f)
